# Lesson 2.2–2.6 — Trajectory、observation schema 与 dataset 转换

本 notebook 按课程所需的顺序，检查原始 simulation trajectory 及其背后的
observation space：

1. 原始 HDF5 结构与单个 timestep 的含义；
2. trajectory 长度与 `(o_t, a_t)` 配对约定；
3. 标准化的 episode 格式及其 timestamp 的来源；
4. observation space，以及原始 state 与展平后的 policy 输入之间的差异；
5. PickCube 的 42 维 observation schema；
6. 控制频率与 action space；
7. 从原始 trajectory 到 LeRobot dataset 的转换。

目标不是跑通一条 pipeline，而是能够对 dataset 中的每一个数字说明：它测量的是什么、
在哪个 frame 下、以何种频率、来自哪个来源。


# robot trajectory dataset


## 2.4 — 原始 HDF5 trajectory 结构

源 trajectory 是一个普通的 HDF5 文件：数组进、数组出，尚未施加任何面向训练的
schema。应把它当作 simulator 的输出记录来读，而不是当作训练 dataset。


In [1]:
from pathlib import Path

# Resolve the dataset directory relative to the REPOSITORY ROOT.
# Paths below are written from the repository root, so run this notebook with the
# repository root as the working directory (the repository's documented convention).
from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
REPO_ROOT = PROJECT_ROOT
DATASET_DIR = REPO_ROOT / "datasets" / "pickcube"

# Keep the Hugging Face cache inside the project: reads must never fall back to the home
# directory (not writable here), and nothing should be written outside the repository.
# This must run BEFORE lerobot is imported -- the `datasets` package snapshots its cache
# location at import time.
import os

HF_CACHE = PROJECT_ROOT / ".cache" / "hf"
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE / "datasets"))


'/home/bowenyuan/Projects/embodied-ai-learning/.cache/hf/datasets'

In [2]:
import h5py

f = h5py.File(
    DATASET_DIR / "random_episode_000.h5",
    "r"
)

print(list(f.keys()))

['actions', 'observations', 'rewards']


In [3]:
t = 0

print("Observation:")
print(f["observations"][t])

print("\nAction:")
print(f["actions"][t])

print("\nReward:")
print(f["rewards"][t])

Observation:
[[ 3.5281047e-02  4.0070224e-01  1.9574760e-02 -1.9186776e+00
   3.7351161e-02  2.3366489e+00  8.0439991e-01  3.9999999e-02
   3.9999999e-02  0.0000000e+00  0.0000000e+00  0.0000000e+00
   0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
   0.0000000e+00  0.0000000e+00  0.0000000e+00  1.2253533e-02
   3.8011339e-02  1.8215224e-01 -1.7688958e-02  9.9980247e-01
   4.2879577e-03  7.9842266e-03  2.6815735e-02 -1.9813180e-03
   2.8893346e-01 -7.4867904e-04  5.3644367e-02  2.0000000e-02
   5.6876123e-01  0.0000000e+00  0.0000000e+00  8.2250267e-01
  -1.3002212e-02  1.5633028e-02 -1.6215225e-01  2.7564414e-02
  -5.5625685e-02  2.6893345e-01]]

Action:
[ 0.17911236  0.9837746   0.1853819  -0.830926   -0.28162387 -0.42015406
 -0.8190485   0.00515126]

Reward:
[0.07952154]


In [4]:
for key in f.keys():
    print(key, f[key].shape, f[key].dtype)

actions (50, 8) float32
observations (50, 1, 42) float32
rewards (50, 1) float32


In [5]:
## 3.2 Understand timestamp

In [6]:
T = f["actions"].shape[0]

print("Trajectory length:", T)

Trajectory length: 50


In [7]:
t = 0

obs_t = f["observations"][t]

action_t = f["actions"][t]

reward_t = f["rewards"][t]


print("obs_t shape:", obs_t.shape)
print("action_t:", action_t)
print("reward_t:", reward_t)

obs_t shape: (1, 42)
action_t: [ 0.17911236  0.9837746   0.1853819  -0.830926   -0.28162387 -0.42015406
 -0.8190485   0.00515126]
reward_t: [0.07952154]


## 创建新的标准化 Robot episode 版本


## 2.3 / 2.4 — 长度、配对与标准化的 episode

本节涉及两个相互独立的问题，值得分开对待：

- **配对约定（pairing convention）**：一个 timestep 贡献 `(o_t, a_t)`，而 `o_{t+1}`
  是 `env.step(a_t)` 产生的。在 `(o_t, a_{t+1})` 上训练会构成 off-by-one 的监督 bug；
- **存储约定（storage convention）**：把展平的 trajectory reshape 为标准化的
  episode，并生成其 timestamps。

下面的 timestamps 由 `np.arange(T) * 0.02` 合成，这假设了 50 Hz。它们不是实测的采集
时间。该环境的真实控制频率是 20 Hz，因此这正是记录在 `notes/progress.md` 中的
20 Hz / 50 Hz 契约缺陷的来源。


In [8]:
import h5py
import numpy as np


src = DATASET_DIR / "random_episode_000.h5"

dst = DATASET_DIR / "random_episode_standard.h5"


with h5py.File(src, "r") as f:

    observations = f["observations"][:]
    actions = f["actions"][:]
    rewards = f["rewards"][:]


# remove simulation batch dimension
observations = np.squeeze(
    observations,
    axis=1
)


T = actions.shape[0]


timestamps = np.arange(T) * 0.02


with h5py.File(dst, "w") as f:

    f.create_dataset(
        "observations",
        data=observations
    )

    f.create_dataset(
        "actions",
        data=actions
    )

    f.create_dataset(
        "rewards",
        data=rewards
    )

    f.create_dataset(
        "timestamps",
        data=timestamps
    )


    metadata = f.create_group(
        "metadata"
    )

    metadata.attrs["task"] = "PickCube-v1"
    metadata.attrs["robot"] = "Panda"
    metadata.attrs["source"] = "ManiSkill"
    metadata.attrs["control_mode"] = "pd_joint_delta_pos"


print("Saved:", dst)

Saved: /home/bowenyuan/Projects/embodied-ai-learning/datasets/pickcube/random_episode_standard.h5


In [9]:
id="5a1f9a"
with h5py.File(dst,"r") as f:

    print(list(f.keys()))

    print(
        dict(
            f["metadata"].attrs
        )
    )

['actions', 'metadata', 'observations', 'rewards', 'timestamps']
{'control_mode': 'pd_joint_delta_pos', 'robot': 'Panda', 'source': 'ManiSkill', 'task': 'PickCube-v1'}


In [10]:
import h5py

path = DATASET_DIR / "random_episode_standard.h5"

with h5py.File(path,"r") as f:

    for key in f.keys():

        print(key)

        if key != "metadata":
            print(" shape:", f[key].shape)
            print(" dtype:", f[key].dtype)

actions
 shape: (50, 8)
 dtype: float32
metadata
observations
 shape: (50, 42)
 dtype: float32
rewards
 shape: (50, 1)
 dtype: float32
timestamps
 shape: (50,)
 dtype: float64


In [11]:
import h5py

path = DATASET_DIR / "random_episode_standard.h5"

f = h5py.File(path, "r")

obs = f["observations"][0]
action = f["actions"][0]

print(obs.shape)
print(action.shape)

f["observations"].shape
f["actions"].shape
obs
action

(42,)
(8,)


array([ 0.17911236,  0.9837746 ,  0.1853819 , -0.830926  , -0.28162387,
       -0.42015406, -0.8190485 ,  0.00515126], dtype=float32)

## 现在检查 Observation Space


## 2.2 — Observation space：原始 state 与展平后的 policy 输入

`obs_mode="state"` 返回一个展平的 tensor，因此这 42 个数字到达时并不带结构。下面的
cell 将它逐层剥开：

- `env.observation_space` 与 `env.unwrapped.observation_space`；
- agent 的 proprioception（`qpos`、`qvel`）与完整的 `get_state()`；
- TCP pose 与任务相关的 `extra` state；
- `_get_obs_state_dict()`，它展示拼接之前未展平的 `agent` + `extra` 分组。

要持续追问：这些字段中哪些是真实机器人能够实际测量的，哪些是 simulator 的
privileged state。


In [12]:
import gymnasium as gym
import mani_skill.envs


env = gym.make(
    "PickCube-v1",
    num_envs=1,
    obs_mode="state",
    control_mode="pd_joint_delta_pos",
)


print(env.observation_space)

obs, info = env.reset(seed=0)

print(type(obs))

2026-09-22 11:37:02,280 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


Box(-inf, inf, (1, 42), float32)
<class 'torch.Tensor'>


In [13]:
obs, info = env.reset(seed=0)

print(obs.shape)
print(obs)

print(env.observation_space)

torch.Size([1, 42])
tensor([[ 3.5281e-02,  4.0070e-01,  1.9575e-02, -1.9187e+00,  3.7351e-02,
          2.3366e+00,  8.0440e-01,  4.0000e-02,  4.0000e-02,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.2254e-02,
          3.8011e-02,  1.8215e-01, -1.7689e-02,  9.9980e-01,  4.2880e-03,
          7.9842e-03,  2.6816e-02, -1.9813e-03,  2.8893e-01, -7.4868e-04,
          5.3644e-02,  2.0000e-02,  5.6876e-01,  0.0000e+00,  0.0000e+00,
          8.2250e-01, -1.3002e-02,  1.5633e-02, -1.6215e-01,  2.7564e-02,
         -5.5626e-02,  2.6893e-01]])
Box(-inf, inf, (1, 42), float32)


In [14]:
print(env.observation_space)
print(env.unwrapped.observation_space)
print(env)

Box(-inf, inf, (1, 42), float32)
Box(-inf, inf, (1, 42), float32)
<TimeLimitWrapper<OrderEnforcing<PickCubeEnv<PickCube-v1>>>>


In [15]:
base_env = env.unwrapped

print(type(base_env))

<class 'mani_skill.envs.tasks.tabletop.pick_cube.PickCubeEnv'>


In [16]:
print(base_env.observation_space)

Box(-inf, inf, (1, 42), float32)


In [17]:
print(base_env.agent)
print(dir(base_env.agent))

['__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_after_init', '_after_loading_articulation', '_agent_idx', '_control_freq', '_control_mode', '_controller_configs', '_default_control_mode', '_load_articulation', '_sensor_configs', 'action_space', 'arm_damping', 'arm_force_limit', 'arm_joint_names', 'arm_stiffness', 'before_simulation_step', 'build_grasp_pose', 'build_separate', 'control_mode', 'controller', 'controllers', 'device', 'disable_self_collisions', 'ee_link_name', 'finger1_link', 'finger1pad_link', 'finger2_link', 'finger2pad_link', 'fix_root_link', 'get_controller_state', 'get_proprioception', 'get_state', 'gripper_damping', 'gripper_force_limit', '

In [18]:
print(base_env.agent.action_space)

Box(-1.0, 1.0, (8,), float32)


In [19]:
proprio = base_env.agent.get_proprioception()

print(type(proprio))
print(proprio)

<class 'dict'>
{'qpos': tensor([[ 0.0353,  0.4007,  0.0196, -1.9187,  0.0374,  2.3366,  0.8044,  0.0400,
          0.0400]]), 'qvel': tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.]])}


In [20]:
state = base_env.agent.get_state()

print(type(state))
print(state)

<class 'dict'>
{'robot_root_pose': Pose(raw_pose=tensor([[-6.1500e-01,  7.2760e-11, -1.4901e-08,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00]])), 'robot_root_vel': tensor([[0., 0., 0.]]), 'robot_root_qvel': tensor([[0., 0., 0.]]), 'robot_qpos': tensor([[ 0.0353,  0.4007,  0.0196, -1.9187,  0.0374,  2.3366,  0.8044,  0.0400,
          0.0400]]), 'robot_qvel': tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.]]), 'controller': {}}


In [21]:
print(base_env.agent.tcp_pose)
print(base_env.agent.tcp_pos)

Pose(raw_pose=tensor([[ 0.0123,  0.0380,  0.1822, -0.0177,  0.9998,  0.0043,  0.0080]]))
tensor([[0.0123, 0.0380, 0.1822]])


In [22]:
obs, info = env.reset(seed=0)

print([x for x in dir(base_env) if "obs" in x.lower()])

['SUPPORTED_OBS_MODES', '_flatten_raw_obs', '_get_obs_agent', '_get_obs_extra', '_get_obs_sensor_data', '_get_obs_state_dict', '_get_obs_with_sensor_data', '_init_raw_obs', '_last_obs', '_obs_mode', 'get_obs', 'obs_mode', 'obs_mode_struct', 'observation_space', 'single_observation_space', 'update_obs_space']


['SUPPORTED_OBS_MODES', '_flatten_raw_obs', '_get_obs_agent', '_get_obs_extra', '_get_obs_sensor_data', '_get_obs_state_dict', '_get_obs_with_sensor_data', '_init_raw_obs', '_last_obs', '_obs_mode', 'get_obs', 'obs_mode', 'obs_mode_struct', 'observation_space', 'single_observation_space', 'update_obs_space']


obs=flatten(agent_state+extra_state)


In [23]:
obs, info = env.reset(seed=0)

raw_obs = base_env._get_obs_state_dict(info)

print(type(raw_obs))

print(raw_obs.keys())

<class 'dict'>
dict_keys(['agent', 'extra'])


In [24]:
print(raw_obs["agent"].keys())

dict_keys(['qpos', 'qvel'])


In [25]:
for k, v in raw_obs["agent"].items():
    print("================")
    print(k)
    print(type(v))
    if hasattr(v, "shape"):
        print("shape:", v.shape)
    print(v)

qpos
<class 'torch.Tensor'>
shape: torch.Size([1, 9])
tensor([[ 0.0353,  0.4007,  0.0196, -1.9187,  0.0374,  2.3366,  0.8044,  0.0400,
          0.0400]])
qvel
<class 'torch.Tensor'>
shape: torch.Size([1, 9])
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.]])


In [26]:
print(raw_obs["extra"].keys())

for k, v in raw_obs["extra"].items():
    print("================")
    print(k)
    print(type(v))
    if hasattr(v, "shape"):
        print("shape:", v.shape)
    print(v)

dict_keys(['is_grasped', 'tcp_pose', 'goal_pos', 'obj_pose', 'tcp_to_obj_pos', 'obj_to_goal_pos'])
is_grasped
<class 'torch.Tensor'>
shape: torch.Size([1])
tensor([False])
tcp_pose
<class 'torch.Tensor'>
shape: torch.Size([1, 7])
tensor([[ 0.0123,  0.0380,  0.1822, -0.0177,  0.9998,  0.0043,  0.0080]])
goal_pos
<class 'torch.Tensor'>
shape: torch.Size([1, 3])
tensor([[ 0.0268, -0.0020,  0.2889]])
obj_pose
<class 'torch.Tensor'>
shape: torch.Size([1, 7])
tensor([[-7.4868e-04,  5.3644e-02,  2.0000e-02,  5.6876e-01,  0.0000e+00,
          0.0000e+00,  8.2250e-01]])
tcp_to_obj_pos
<class 'torch.Tensor'>
shape: torch.Size([1, 3])
tensor([[-0.0130,  0.0156, -0.1622]])
obj_to_goal_pos
<class 'torch.Tensor'>
shape: torch.Size([1, 3])
tensor([[ 0.0276, -0.0556,  0.2689]])


# PickCube 机器人 observation schema

## 概览

在 ManiSkill 中，环境返回的原始 observation 是一个展平的 tensor：

\[
o_t \in R^{42}
\]

尽管 observation 呈现为 42 维向量，它由多个语义分量组成：

\[
Observation =
Agent\ State + Extra\ Task\ State
\]

展平之前的结构化 observation 是：

```text
Observation

├── agent
│   ├── robot_qpos
│   └── robot_qvel
│
└── extra
    ├── tcp_pose
    ├── goal_pos
    ├── obj_pose
    ├── tcp_to_obj_pos
    ├── obj_to_goal_pos
    └── is_grasped

In [27]:
print(base_env.agent.arm_joint_names)
print(base_env.agent.gripper_joint_names)

['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']
['panda_finger_joint1', 'panda_finger_joint2']


# 测试 Lerobot


## 2.5 / 2.6 — 控制频率、action space 与 LeRobot

最后一步把已检查的 trajectory 与面向训练的 dataset 格式连接起来。这里记录的两点值得
注意：

- `control_freq` 是环境的真实控制频率，dataset 声明的 FPS 必须与这个数字一致；
- `env.action_space` 是单个 `Box(-1, 1, (8,), float32)`，但这八个通道**并非同一个
  语义空间**：机械臂通道是 joint-position delta，gripper 通道则是 absolute
  position 目标。


In [28]:
import lerobot
import lerobot.datasets
print(lerobot.__version__)

0.6.1


In [29]:
import inspect
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print(inspect.signature(LeRobotDataset.create))

(repo_id: str, fps: int, features: dict, root: str | pathlib.Path | None = None, robot_type: str | None = None, use_videos: bool = True, tolerance_s: float = 0.0001, image_writer_processes: int = 0, image_writer_threads: int = 0, video_backend: str | None = None, batch_encoding_size: int = 1, rgb_encoder: lerobot.configs.video.RGBEncoderConfig | None = None, depth_encoder: lerobot.configs.video.DepthEncoderConfig | None = None, metadata_buffer_size: int = 10, streaming_encoding: bool = False, encoder_queue_maxsize: int = 30, encoder_threads: int | None = None, video_files_size_in_mb: int | None = None, data_files_size_in_mb: int | None = None) -> 'LeRobotDataset'


In [30]:
print(base_env.control_freq)

20


In [31]:
print(env.action_space)
print(env.unwrapped.agent.controller)

Box(-1.0, 1.0, (8,), float32)
CombinedController(dof=8, active_joints=9
    arm: PDJointPosController(dof=7, active_joints=7, joints=(panda_joint1, panda_joint2, panda_joint3, panda_joint4, panda_joint5, panda_joint6, panda_joint7))
    gripper: PDJointPosMimicController(dof=1, active_joints=2, mimic_to_control_joint_map={
        panda_finger_joint2: panda_finger_joint1,
    })
)


In [32]:
print(actions[:5])

[[ 0.17911236  0.9837746   0.1853819  -0.830926   -0.28162387 -0.42015406
  -0.8190485   0.00515126]
 [ 0.8806315  -0.9077806   0.01525491  0.08429879  0.4479404  -0.86594576
  -0.38341427 -0.4729761 ]
 [ 0.5689431   0.62343407  0.5606486   0.8035238   0.5462453   0.147302
  -0.7543964   0.09960714]
 [-0.67252946 -0.5255221   0.19235007  0.9518804   0.31535336 -0.889083
  -0.26096365 -0.32041526]
 [ 0.8688718  -0.09353677 -0.74594826 -0.54997814  0.50250185  0.4947652
  -0.5517533   0.9721042 ]]


In [33]:
base_env = env.unwrapped

print("Control mode:", base_env.control_mode)

controller = base_env.agent.controller
print("Controller type:", type(controller))
print("Controller:", controller)

if hasattr(controller, "configs"):
    for name, config in controller.configs.items():
        print(f"\n{name}:")
        print(config)

Control mode: pd_joint_delta_pos
Controller type: <class 'mani_skill.agents.controllers.base_controller.CombinedController'>
Controller: CombinedController(dof=8, active_joints=9
    arm: PDJointPosController(dof=7, active_joints=7, joints=(panda_joint1, panda_joint2, panda_joint3, panda_joint4, panda_joint5, panda_joint6, panda_joint7))
    gripper: PDJointPosMimicController(dof=1, active_joints=2, mimic_to_control_joint_map={
        panda_finger_joint2: panda_finger_joint1,
    })
)

arm:
PDJointPosControllerConfig(joint_names=['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7'], lower=-0.1, upper=0.1, stiffness=1000.0, damping=100.0, force_limit=100, friction=0.0, use_delta=True, use_target=False, interpolate=False, normalize_action=True, drive_mode='force')

gripper:
PDJointPosMimicControllerConfig(joint_names=['panda_finger_joint1', 'panda_finger_joint2'], lower=-0.01, upper=0.04, stiffness=1000.0, damping=100.0, force_l

## 2.4.6 — 验证 trajectory 文件（由 scripts 整合而来）

下面的检查原先由三个独立脚本完成，现已退役到 `archive/lesson_2_superseded/`。它们属于
本节，因为它们是理解数据的一部分，而不是独立的工具：

- `validate_maniskill_rollout.py` — schema、长度、连续性、episode 语义
- `dataset_report.py` — 概览、取值范围、平滑性（其函数位于
  `scripts/pipeline/reporter.py`，在转换过程中运行）
- `compare_random_datasets.py` — 比较两个 dataset 变体

`datasets/pickcube/` 下存在三个文件，它们构成一条**谱系（lineage）**，而不是三种可
互相替代的方案：

| 文件 | Observations | Extra fields | 作用 |
|---|---|---|---|
| `maniskill_random_rollout.h5` | `(50, 42)` | `next_observations`, `success`, `terminated`, `truncated`, `elapsed_steps`, `is_*` | 原始 collector 输出 |
| `random_episode_000.h5` | `(50, 1, 42)` | none | 中间产物，仍带有 env 的 batch 轴 |
| `random_episode_standard.h5` | `(50, 42)` | `timestamps`, `metadata` group | pipeline 消费的标准化 fixture |

中间那个文件保留了 `num_envs` 轴。这就是标准化时必须 squeeze 它的原因；一个"看起来
相同"的 dataset 仍可能多出一个维度。


In [34]:
import h5py
import numpy as np
from pathlib import Path

PICKCUBE_DIR = PROJECT_ROOT / "datasets" / "pickcube"

FILES = {
    "raw collector output": PICKCUBE_DIR / "maniskill_random_rollout.h5",
    "intermediate (batch axis)": PICKCUBE_DIR / "random_episode_000.h5",
    "standardized fixture": PICKCUBE_DIR / "random_episode_standard.h5",
}


def describe_h5(path):
    """Report every dataset and group in an HDF5 file, plus root attributes."""
    with h5py.File(path, "r") as handle:
        print(f"root attributes: {dict(handle.attrs) or '(none)'}")
        for key in handle:
            obj = handle[key]
            if isinstance(obj, h5py.Dataset):
                print(f"  {key:<20} shape={str(obj.shape):<14} dtype={obj.dtype}")
            else:
                print(f"  {key:<20} GROUP attrs={dict(obj.attrs)}")


for label, path in FILES.items():
    print(f"\n===== {label} =====")
    print(path.name)
    describe_h5(path)


===== raw collector output =====
maniskill_random_rollout.h5
root attributes: {'control_mode': 'pd_joint_delta_pos', 'environment': 'PickCube-v1', 'obs_mode': 'state', 'policy': 'random', 'seed': np.int64(0)}
  actions              shape=(50, 8)        dtype=float32
  elapsed_steps        shape=(50,)          dtype=int32
  is_grasped           shape=(50, 1)        dtype=bool
  is_obj_placed        shape=(50, 1)        dtype=bool
  is_robot_static      shape=(50, 1)        dtype=bool
  next_observations    shape=(50, 42)       dtype=float32
  observations         shape=(50, 42)       dtype=float32
  rewards              shape=(50, 1)        dtype=float32
  success              shape=(50, 1)        dtype=bool
  terminated           shape=(50, 1)        dtype=bool
  truncated            shape=(50, 1)        dtype=bool

===== intermediate (batch axis) =====
random_episode_000.h5
root attributes: (none)
  actions              shape=(50, 8)        dtype=float32
  observations         shape=

### Schema 与长度检查

在任何数字有意义之前，必须先满足两个性质：

1. 描述同一条 trajectory 的每个字段都具有**相同的长度**；
2. `observations[t]` 是施加 `actions[t]` *之前*的 state，即
   `next_observations[t] == observations[t+1]`。


In [35]:
with h5py.File(FILES["raw collector output"], "r") as handle:
    observations = handle["observations"][:]
    actions = handle["actions"][:]
    rewards = handle["rewards"][:]
    next_observations = handle["next_observations"][:]
    success = handle["success"][:]
    terminated = handle["terminated"][:]
    truncated = handle["truncated"][:]
    elapsed_steps = handle["elapsed_steps"][:]

print("===== Shapes =====")
print(f"observations      {observations.shape} {observations.dtype}")
print(f"actions           {actions.shape} {actions.dtype}")
print(f"rewards           {rewards.shape} {rewards.dtype}")
print(f"next_observations {next_observations.shape} {next_observations.dtype}")

print("\n===== Length consistency =====")
lengths = {name: len(arr) for name, arr in [
    ("observations", observations), ("actions", actions), ("rewards", rewards),
    ("next_observations", next_observations), ("success", success),
    ("terminated", terminated), ("truncated", truncated), ("elapsed_steps", elapsed_steps),
]}
print(lengths)
print("[PASS] all field lengths equal" if len(set(lengths.values())) == 1 else "[FAIL] lengths differ")

print("\n===== T actions vs T+1 states =====")
print(f"T actions            : {len(actions)}")
print(f"next_observations[T-1] is the terminal state o_T")
print("next_observations[t] == observations[t+1] for t in [0, T-2]:",
      np.allclose(next_observations[:-1], observations[1:], atol=1e-6))
print("observations has T entries, not T+1: the terminal state lives only in next_observations.")

print("\n===== Numeric health =====")
for name, arr in [("observations", observations), ("actions", actions), ("rewards", rewards)]:
    finite = np.isfinite(arr).all()
    print(f"{name:<14} finite={finite}  min={arr.min():+.4f}  max={arr.max():+.4f}")

print("\n===== Episode semantics =====")
print(f"terminated count: {int(terminated.sum())}")
print(f"truncated  count: {int(truncated.sum())}")
print(f"success    count: {int(success.sum())}")
print(f"final success: {bool(success[-1, 0])} | final truncated: {bool(truncated[-1, 0])}")
print(f"elapsed_steps continuous (+1 each step): {bool(np.all(np.diff(elapsed_steps) == 1))}")

===== Shapes =====
observations      (50, 42) float32
actions           (50, 8) float32
rewards           (50, 1) float32
next_observations (50, 42) float32

===== Length consistency =====
{'observations': 50, 'actions': 50, 'rewards': 50, 'next_observations': 50, 'success': 50, 'terminated': 50, 'truncated': 50, 'elapsed_steps': 50}
[PASS] all field lengths equal

===== T actions vs T+1 states =====
T actions            : 50
next_observations[T-1] is the terminal state o_T
next_observations[t] == observations[t+1] for t in [0, T-2]: True
observations has T entries, not T+1: the terminal state lives only in next_observations.

===== Numeric health =====
observations   finite=True  min=-1.9187  max=+2.4367
actions        finite=True  min=-0.9990  max=+0.9915
rewards        finite=True  min=+0.0116  max=+0.0808

===== Episode semantics =====
terminated count: 0
truncated  count: 1
success    count: 0
final success: False | final truncated: True
elapsed_steps continuous (+1 each step): Tr

### Action 健康度：range 与 smoothness

range 与 smoothness 回答的是不同的问题，二者对 imitation data 都很重要：

- **range**：取值是否落在文档给出的边界内？
- **smoothness**：action 在相邻 step 之间的跳变有多大？

随机 action 在 range 上完全合法，却极不平滑。这种组合正是一条 pipeline fixture 的
特征，而不是 demonstration 的特征。


In [36]:
print("===== Action range =====")
print(f"action space documented range: [-1, 1]")
print(f"observed min: {actions.min():+.4f}  max: {actions.max():+.4f}")
print("[PASS] all actions within [-1, 1]" if (actions >= -1).all() and (actions <= 1).all() else "[FAIL] out of range")

print("\n===== Action range per channel =====")
channel_names = [f"arm_joint{i+1}" for i in range(7)] + ["gripper"]
for index, name in enumerate(channel_names):
    channel = actions[:, index]
    print(f"  {name:<12} min={channel.min():+.4f} max={channel.max():+.4f} std={channel.std():.4f}")

print("\n===== Smoothness: consecutive-step change =====")
delta = np.abs(np.diff(actions, axis=0))
print(f"mean |da|: {delta.mean():.4f}")
print(f"p95  |da|: {np.percentile(delta, 95):.4f}")
print(f"max  |da|: {delta.max():.4f}")
print()
print("An expert trajectory would show small, correlated steps.")
print("Steps of this size mean consecutive actions are nearly independent: the data is random.")

===== Action range =====
action space documented range: [-1, 1]
observed min: -0.9990  max: +0.9915
[PASS] all actions within [-1, 1]

===== Action range per channel =====
  arm_joint1   min=-0.9828 max=+0.9605 std=0.5528
  arm_joint2   min=-0.9680 max=+0.9750 std=0.5739
  arm_joint3   min=-0.9872 max=+0.9701 std=0.6114
  arm_joint4   min=-0.9345 max=+0.9868 std=0.6031
  arm_joint5   min=-0.9847 max=+0.9800 std=0.5928
  arm_joint6   min=-0.9965 max=+0.9915 std=0.5947
  arm_joint7   min=-0.9248 max=+0.9177 std=0.5536
  gripper      min=-0.9990 max=+0.9264 std=0.5843

===== Smoothness: consecutive-step change =====
mean |da|: 0.6625
p95  |da|: 1.6254
max  |da|: 1.9880

An expert trajectory would show small, correlated steps.
Steps of this size mean consecutive actions are nearly independent: the data is random.


### 比较两个 dataset 变体

`random_episode_000.h5` 与 `maniskill_random_rollout.h5` 都由随机 collector 生成，但
二者不可互换：其中一个带有 env 的 batch 轴，而且两者都没有 success 记录。同时加载
两者并比较 shape 与统计量，才能把"shape 相等"与"数据兼容"区分开。

真正重要的检查：带 batch 轴的文件必须先 squeeze，才能与另一个文件进行比较；而且两者
**并不是**同一条 trajectory。


In [37]:
def load_arrays(path, keys):
    with h5py.File(path, "r") as handle:
        return {key: handle[key][:] for key in keys}


first = load_arrays(FILES["raw collector output"], ["observations", "actions"])
second = load_arrays(FILES["intermediate (batch axis)"], ["observations", "actions"])

print("===== Shape comparison =====")
print(f"raw           observations {first['observations'].shape}  actions {first['actions'].shape}")
print(f"intermediate  observations {second['observations'].shape}  actions {second['actions'].shape}")
print()
print("same observation shape:", first["observations"].shape == second["observations"].shape)
print("=> the intermediate file carries an extra num_envs axis of 1")

squeezed = np.squeeze(second["observations"], axis=1)
print("after squeeze:", squeezed.shape)

print("\n===== Are they the same trajectory? =====")
print("same actions:", np.allclose(first["actions"], second["actions"], atol=1e-6))
print("same observations after squeeze:",
      np.allclose(first["observations"], squeezed, atol=1e-6))
print()
print("Two files can have identical shapes and different contents;")
print("two files can hold the same arrays in different layouts.")
print("Neither shape nor one matching array is proof of equivalence.")

print("\n===== Action statistics, side by side =====")
for label, arrays in (("raw", first), ("intermediate", second)):
    channel = arrays["actions"][:, 0]
    print(f"  {label:<14} action[0] mean={channel.mean():+.4f} std={channel.std():.4f}")

===== Shape comparison =====
raw           observations (50, 42)  actions (50, 8)
intermediate  observations (50, 1, 42)  actions (50, 8)

same observation shape: False
=> the intermediate file carries an extra num_envs axis of 1
after squeeze: (50, 42)

===== Are they the same trajectory? =====
same actions: False
same observations after squeeze: False

Two files can have identical shapes and different contents;
two files can hold the same arrays in different layouts.
Neither shape nor one matching array is proof of equivalence.

===== Action statistics, side by side =====
  raw            action[0] mean=-0.1069 std=0.5528
  intermediate   action[0] mean=-0.0118 std=0.6168


### 这套验证不能证明什么

上面的检查全部通过时，数据仍可能对训练毫无用处。这些检查确立的是：

- schema 是一致的；
- 数组是有限的且落在取值范围内；
- episode 边界与配对约定成立。

它们**不能**确立 action 是有意义的。success 计数为 `0`，且 action delta 近似相互
独立，因此这条 trajectory 无法训练出 policy。*format valid* 与 *data useful* 的
区别，正是 2.6 所划分的，也是 2.9 必须产出真实 demonstrations 的原因。


### 这些检查现在位于何处

`dataset_report.py` 中的报告与质量汇总逻辑已移入转换 pipeline
（`scripts/pipeline/reporter.py`），因此转换后的 dataset 会在转换过程中被报告，而
不再依赖单独的脚本。这三个脚本的独立版本已退役到
`archive/lesson_2_superseded/`，退役原因记录在 `archive/README.md`。检查本身仍然是
上面的这些 cell。
